# Gage and Adhesive Selection: The Precision Specification
### *Modern Strain Gage Technology* — Companion Notebook

**The bond beneath the gage is the least glamorous part of the measurement system and the one most responsible for failures.**

Companion notebook to Chapter 42 — Gage and Adhesive Selection: The Precision Specification.

The gage is only as good as the bond beneath it. A disbond, a void, or an adhesive that creeps under sustained load corrupts the strain transfer and produces errors that cannot be separated from real mechanical signal. Getting the adhesive right is not a detail — it is a prerequisite, and one of the decisions that separates a 0.5 %-class transducer from a 0.02 %-class one.

The selection space is defined by four variables: **maximum operating temperature**, **moisture resistance**, **fatigue life**, and **installation complexity**. *Cyanoacrylate* (CA) is the workhorse of laboratory testing — fast, easy, and adequate for short, room-temperature tests — but it has hard limits: above roughly 65–80 °C it softens, and under cyclic load its brittle bond layer microcracks. *Epoxy* systems are more demanding to install but far more capable at elevated temperature and under fatigue. *Ceramic cement* pushes the ceiling to about 1000 °C at the cost of fatigue performance and installation ease.

**This notebook demonstrates:**
1. An **adhesive selection guide** — five adhesive families compared on temperature limit, moisture resistance, fatigue rating, and installation difficulty.
2. A **temperature-envelope plot** that makes visible why operating temperature is the first screen in any adhesive choice.

In [ ]:
# Colab setup — dependencies are preinstalled on Colab; uncomment to install locally:
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Deterministic, legible rendering. This notebook uses no randomness, so there is
# nothing to seed; every figure below is fully reproducible from the fixed data.
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — Adhesive Selection Guide

The table below is a decision matrix, not a specification sheet. The ratings — *Poor, Medium, Good, Excellent* — represent relative performance within each column, not absolute pass/fail thresholds. A "Poor" moisture rating does not mean the adhesive fails the moment it meets humidity; it means the adhesive degrades faster than the alternatives and should not be chosen for outdoor or high-humidity service.

Two decisions drive most adhesive choices. First, **operating temperature**: if the environment exceeds 65 °C, cyanoacrylate is eliminated immediately; above 150 °C, room-temperature-curing epoxy drops out; above 260 °C, only ceramic cement and the polyimide film systems remain. Second, **dynamic loading**: cyanoacrylate has limited fatigue performance because its bond layer is brittle, so elevated-cure epoxy is the workhorse for high-cycle fatigue testing. Installation difficulty is considered last — it costs the engineer time, but it does not degrade the measurement if the bonding procedure is followed correctly.

We print the guide as a table first, then plot the operating-temperature envelope so the temperature screen is visible at a glance.

In [ ]:
# --- Adhesive selection guide -------------------------------------------------
# A decision matrix for the four variables that drive most bonding choices:
# maximum operating temperature, moisture resistance, fatigue rating, and
# installation difficulty. Ratings are RELATIVE within each column, not
# absolute limits — always confirm against the manufacturer's datasheet.

CA_RECOMMENDED_LIMIT_C = 65   # conservative upper service limit for cyanoacrylate

# T_max is the approximate maximum continuous operating temperature in °C.
adhesives = {
    'Cyanoacrylate (CA)':    {'T_max': 80,   'moisture': 'Poor', 'fatigue': 'Poor',      'install': 'Easy'},
    'Epoxy (room cure)':     {'T_max': 150,  'moisture': 'Good', 'fatigue': 'Good',      'install': 'Medium'},
    'Epoxy (elevated cure)': {'T_max': 260,  'moisture': 'Good', 'fatigue': 'Excellent', 'install': 'Hard'},
    'Ceramic cement':        {'T_max': 1000, 'moisture': 'Good', 'fatigue': 'Poor',      'install': 'Hard'},
    'Polyimide film tape':   {'T_max': 300,  'moisture': 'Good', 'fatigue': 'Medium',    'install': 'Easy'},
}


def print_selection_guide(adhesives):
    """Print the adhesive decision matrix as an aligned, fixed-width text table.

    Each row shows the adhesive's maximum operating temperature (°C) and its
    relative moisture, fatigue, and installation ratings. Intended as a first
    screen, not a substitute for the manufacturer's bonding datasheet.
    """
    header = f"{'Adhesive':<24}{'T_max':>8}{'Moisture':>11}{'Fatigue':>12}{'Install':>10}"
    print("ADHESIVE SELECTION GUIDE")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for name, props in adhesives.items():
        t_max = f"{props['T_max']}°C"
        print(f"{name:<24}{t_max:>8}{props['moisture']:>11}"
              f"{props['fatigue']:>12}{props['install']:>10}")


print_selection_guide(adhesives)
print()
print(f"Rule of thumb: never use CA above ~{CA_RECOMMENDED_LIMIT_C}°C or in dynamic "
      f"fatigue testing — its bond layer is brittle and hygroscopic.")

---
## 2 — The Temperature Envelope

The table lists a temperature limit for each adhesive, but a number in a column is easy to skim past. The plot below makes the first screening decision physical: each bar shows the band of temperature over which the adhesive is usable, from ambient up to its maximum, and its color reports the fatigue rating from the table. Read vertically at any target temperature and only the bars that reach that far are candidates; the dashed marker at 65 °C is the line beyond which cyanoacrylate should not be trusted. The logarithmic axis keeps the room-temperature laboratory adhesives and the 1000 °C ceramic cement legible on the same figure.

This chart is displayed inline only — it is not saved to the book's figure set — so you can freely edit the data or styling without affecting the manuscript.

In [ ]:
# --- Operating-temperature envelope -------------------------------------------
# Each bar spans from room temperature to the adhesive's maximum operating
# temperature and is colored by its fatigue rating, so both selection axes —
# temperature reach and fatigue capability — are readable at once. The x-axis
# is logarithmic because the range covers more than a decade (80 °C to 1000 °C).

ROOM_TEMP_C = 20   # every bond begins at ambient; use it as the left end of each bar

# Ratings ordered worst-to-best; reused for both bar color and legend order.
FATIGUE_COLORS = {
    'Poor':      '#C44E52',
    'Medium':    '#DD8452',
    'Good':      '#55A868',
    'Excellent': '#4C72B0',
}


def plot_temperature_envelope(adhesives):
    """Plot each adhesive's usable temperature span, colored by fatigue rating.

    Bars run from ROOM_TEMP_C to the adhesive's maximum operating temperature on
    a log x-axis. A dashed marker at the cyanoacrylate limit (CA_RECOMMENDED_LIMIT_C)
    shows the threshold that eliminates CA from most elevated-temperature work.
    Returns the Matplotlib Axes so the caller can annotate or save it.
    """
    items = sorted(adhesives.items(), key=lambda kv: kv[1]['T_max'])
    names = [name for name, _ in items]
    y = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for yi, (_, props) in zip(y, items):
        ax.barh(yi, props['T_max'] - ROOM_TEMP_C, left=ROOM_TEMP_C,
                color=FATIGUE_COLORS[props['fatigue']], alpha=0.9)
        ax.text(props['T_max'] * 1.07, yi, f"{props['T_max']}°C",
                va='center', ha='left', fontsize=9)

    ax.axvline(CA_RECOMMENDED_LIMIT_C, color='0.35', linestyle='--', linewidth=1.3)
    ax.text(CA_RECOMMENDED_LIMIT_C * 0.95, len(names) - 0.4, 'CA limit 65°C',
            color='0.30', fontsize=8, rotation=90, va='top', ha='right')

    ax.set_xscale('log')
    ax.set_yticks(y)
    ax.set_yticklabels(names)
    ax.set_xlabel('Maximum operating temperature (°C, log scale)')
    ax.set_title('Adhesive operating-temperature envelope')
    ax.set_xlim(ROOM_TEMP_C, 6000)
    ax.grid(axis='x', which='both', alpha=0.3)

    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=FATIGUE_COLORS[rating])
                      for rating in FATIGUE_COLORS]
    ax.legend(legend_handles, list(FATIGUE_COLORS), title='Fatigue rating',
              loc='lower right', fontsize=8, title_fontsize=8)
    return ax


plot_temperature_envelope(adhesives)
plt.tight_layout()
plt.show()

---
## Where to take this next

This notebook is a starting point. A few concrete extensions:

1. **Turn the matrix into a scoring function.** Give an application a temperature, a humidity level, and a fatigue-cycle target, then write a function that filters the `adhesives` dict to the systems that pass all three and ranks the survivors by installation ease. That converts the table from a lookup into a recommender.
2. **Add the gage-alloy layer.** Chapter 42 pairs adhesive choice with grid alloy (constantan, Karma/EMC, isoelastic, Pt-W). Build a second dict from Table 42.1 and cross-check which alloy/adhesive combinations are self-consistent — a fatigue-rated Karma gage bonded with brittle CA is a contradiction the code can flag automatically.
3. **Model the bondline trade-off.** The chapter argues for a very thin (0.0002 in) bondline to limit creep and reinforcement. Add a simple strain-transfer or reinforcement model as a function of bondline thickness and adhesive modulus, and plot the error a thick, filled epoxy would introduce on a thin bending element.